# Train the contact-sharing classifier (P1a)

Fine-tunes `bert-base-uncased` to give each chat message a score: the probability that it
shares or asks for off-platform contact info. Model: design doc §5. Evaluation: §6.

**Before running**
1. Upload the repo's `data/dataset_v1/rows/` folder (3 parquet files, ~17 MB) to Google Drive
   as `MyDrive/chat-safety/dataset_v1/rows/`.
2. Runtime → Change runtime type → **T4 GPU**.
3. Runtime → Run all. Expect roughly 1–2 hours; each epoch prints its time.

**What it writes** (to Drive, so it survives a Colab disconnect)
```
MyDrive/chat-safety/runs/bert_v1/
  model/               the best epoch's weights + tokenizer
  scores/val.parquet   row_id, score  (score = P(contact), 0 to 1)
  scores/test.parquet
  run_info.json        settings + val PR AUC per epoch
```

**After:** copy `scores/` into the repo as `outputs/bert_v1/` and run
```
uv run python -m modeling.eval.evaluate --model outputs/bert_v1 --baseline outputs/regex --split val --out results/p1a/bert_v1
```
Test scores are written now so the GPU isn't needed again, but don't evaluate `--split test`
until the model and threshold are frozen (design doc §6.4).

## 1. Settings

In [ ]:
# Set to e.g. 2000 for a dry run on a small sample of each split (a few minutes).
# Dry runs write to runs/dry_run so they never overwrite a real run.
LIMIT = None

# Paths on Google Drive
DRIVE_DIR = "/content/drive/MyDrive/chat-safety"
DATA_DIR = f"{DRIVE_DIR}/dataset_v1/rows"
RUN_DIR = f"{DRIVE_DIR}/runs/{'dry_run' if LIMIT else 'bert_v1'}"

# Model
MODEL_NAME = "bert-base-uncased"
# The context builder's speaker and channel tags, each added as one new token.
# Must match SPECIAL_TOKENS in src/modeling/context.py (test_notebook.py checks this).
SPECIAL_TOKENS = [
    "[SELF]",                                                         # the target's author
    "[U1]", "[U2]", "[U3]", "[U4]", "[U5]", "[U6]", "[U7]", "[U8]", "[U9]",  # other speakers
    "[UOTHER]",                                                       # 10th+ other speaker
    "[DM]", "[GROUP]",                                                # channel type
]
MAX_LEN = 512  # BERT's limit. Our rows are at most 256 tokens, so nothing gets cut.

# Training: the standard BERT fine-tuning recipe
EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.06  # learning rate ramps up over the first 6% of steps, then decays to 0
MAX_GRAD_NORM = 1.0     # gradient clipping
SCORE_BATCH_SIZE = 128  # scoring needs no gradients, so bigger batches fit
SEED = 0

## 2. Setup

Colab already has everything installed: torch, transformers, pandas, scikit-learn.

In [ ]:
import json
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import drive
from sklearn.metrics import average_precision_score, roc_curve
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)

drive.mount("/content/drive")
Path(f"{RUN_DIR}/scores").mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_fp16 = device.type == "cuda"  # half precision: ~2-3x faster on a T4
print("device:", torch.cuda.get_device_name() if use_fp16 else "CPU (too slow for a full run)")

## 3. Data

One row per chat message. The columns used here:

| column | what it is |
|---|---|
| `row_id` | unique id, used to join scores back to labels in `evaluate.py` |
| `context_text` | channel tag + the messages before the target, each starting with a speaker tag |
| `target_text` | the message being scored, starting with `[SELF]` |
| `label` | 1 = shares or asks for contact info, 0 = doesn't |

In [ ]:
COLUMNS = ["row_id", "context_text", "target_text", "label"]

data = {}
for split in ("train", "val", "test"):
    df = pd.read_parquet(f"{DATA_DIR}/{split}.parquet", columns=COLUMNS)
    if LIMIT:
        df = df.sample(n=min(LIMIT, len(df)), random_state=SEED)
    data[split] = df.reset_index(drop=True)
    print(f"{split:>5}: {len(df):>7,} rows, {df.label.mean():.1%} positive")

example = data["val"][data["val"].label == 1].iloc[0]
print("\ncontext:", example.context_text)
print("target: ", example.target_text)

## 4. Tokenizer

BERT reads one sequence. A (context, target) pair is packed as

```
[CLS] context tokens [SEP] target tokens [SEP]
  0      0  ...  0     0     1  ...  1     1      <- segment id (token_type_ids)
```

The `[SEP]` tokens and the segment ids both mark where the target starts. Each token's input
vector = word embedding + position embedding + segment embedding. The tokenizer builds all of
this when called as `tokenizer(context, target)`.

The 13 tags are added as new tokens, so `[U1]` is one token instead of `[`, `u`, `##1`, `]`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
n_added = tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
print(f"vocabulary: {len(tokenizer) - n_added:,} + {n_added} tags = {len(tokenizer):,}")

# The example above as BERT sees it: token/segment
enc = tokenizer(example.context_text, example.target_text)
tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"])
print(" ".join(f"{t}/{s}" for t, s in zip(tokens, enc["token_type_ids"])))

In [ ]:
def encode(df):
    """Token ids, segment ids, and label for every row."""
    enc = tokenizer(list(df.context_text), list(df.target_text),
                    truncation=True, max_length=MAX_LEN)
    return [
        {"input_ids": ids, "token_type_ids": segments, "attention_mask": mask, "labels": int(y)}
        for ids, segments, mask, y in zip(
            enc["input_ids"], enc["token_type_ids"], enc["attention_mask"], df.label)
    ]


encoded = {split: encode(df) for split, df in data.items()}
lengths = [len(row["input_ids"]) for row in encoded["train"]]
print(f"train tokens per row: mean {np.mean(lengths):.0f}, max {max(lengths)}")

## 5. Batches

Rows in a batch get padded to the batch's longest row. Grouping rows of similar length keeps
that padding (wasted compute) small: shuffle, sort by length, cut into batches, then shuffle the
batch order. Shuffling first means rows of equal length land in different batches each epoch.

In [ ]:
collate = DataCollatorWithPadding(tokenizer)  # pads one batch and turns it into tensors


def make_batches(rows, batch_size, rng=None):
    """Lists of row indices, grouped by length. Shuffled if `rng` is given (training)."""
    lengths = np.array([len(row["input_ids"]) for row in rows])
    order = rng.permutation(len(rows)) if rng is not None else np.arange(len(rows))
    order = order[np.argsort(lengths[order], kind="stable")]  # by length, ties keep their order
    batches = [order[i:i + batch_size].tolist() for i in range(0, len(order), batch_size)]
    if rng is not None:
        rng.shuffle(batches)
    return batches


def loader(rows, batches):
    return DataLoader(rows, batch_sampler=batches, collate_fn=collate)

## 6. Model

`BertForSequenceClassification` = the pretrained BERT encoder + a classification head on the
`[CLS]` token:

```
[CLS] vector (768) → pooler: Linear 768→768 + tanh   (pretrained)
                   → dropout 0.1
                   → classifier: Linear 768→2        (new, random start, 1,538 parameters)
                   → softmax → score = P(class 1)
```

BERT's pretraining heads (masked-word and next-sentence prediction) are not loaded. The load
message lists them as unused, and lists `classifier.weight` / `classifier.bias` as newly
initialized. Both are expected. All ~109.5M parameters are trained.

In [ ]:
model = BertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.resize_token_embeddings(len(tokenizer))  # 13 new embedding rows, one per tag
model.to(device)

print(model.classifier)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Scoring

The score is the softmax probability of class 1. It's returned in the original row order.

In [ ]:
@torch.no_grad()
def score(rows):
    """P(contact) for every row."""
    model.eval()
    scores = np.zeros(len(rows))
    batches = make_batches(rows, SCORE_BATCH_SIZE)
    for indices, batch in zip(batches, tqdm(loader(rows, batches), total=len(batches),
                                            desc="scoring", leave=False)):
        batch.pop("labels")
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            logits = model(**batch.to(device)).logits
        scores[indices] = logits.float().softmax(dim=-1)[:, 1].cpu().numpy()
    return scores

## 8. Train

Plain PyTorch loop. Each step: forward → cross-entropy loss → backward → clip → update. After
each epoch, score val. The epoch with the best **val PR AUC** is kept (design doc §6.2: used
only to pick the checkpoint), saved to Drive.

fp16 needs a `GradScaler`: it multiplies the loss before `backward()` so small gradients don't
round to zero in half precision, then divides them back before the update.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * math.ceil(len(encoded["train"]) / BATCH_SIZE)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=int(WARMUP_FRACTION * total_steps), num_training_steps=total_steps)
scaler = torch.amp.GradScaler(device.type, enabled=use_fp16)
rng = np.random.default_rng(SEED)

history = []
best_epoch, best_pr_auc = None, -1.0

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    model.train()
    batches = make_batches(encoded["train"], BATCH_SIZE, rng)
    progress = tqdm(loader(encoded["train"], batches), total=len(batches), desc=f"epoch {epoch}")
    for step, batch in enumerate(progress):
        labels = batch.pop("labels").to(device)
        with torch.autocast(device.type, dtype=torch.float16, enabled=use_fp16):
            logits = model(**batch.to(device)).logits
        loss = F.cross_entropy(logits.float(), labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)  # back to true gradients, so clipping sees real sizes
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        if step % 50 == 0:
            progress.set_postfix(loss=f"{loss.item():.4f}")

    val_pr_auc = average_precision_score(data["val"].label, score(encoded["val"]))
    minutes = (time.time() - start) / 60
    history.append({"epoch": epoch, "val_pr_auc": val_pr_auc, "minutes": round(minutes, 1)})
    print(f"epoch {epoch}: val PR AUC {val_pr_auc:.4f} ({minutes:.0f} min)")

    if val_pr_auc > best_pr_auc:
        best_epoch, best_pr_auc = epoch, val_pr_auc
        model.save_pretrained(f"{RUN_DIR}/model")
        tokenizer.save_pretrained(f"{RUN_DIR}/model")
        print(f"  saved as best to {RUN_DIR}/model")

## 9. Score val and test with the best epoch

In [ ]:
model = BertForSequenceClassification.from_pretrained(f"{RUN_DIR}/model").to(device)

scores = {}
for split in ("val", "test"):
    scores[split] = score(encoded[split])
    out = pd.DataFrame({"row_id": data[split].row_id, "score": scores[split]})
    out.to_parquet(f"{RUN_DIR}/scores/{split}.parquet", index=False)
    print(f"wrote {RUN_DIR}/scores/{split}.parquet ({len(out):,} rows)")

settings = {name: globals()[name] for name in (
    "MODEL_NAME", "MAX_LEN", "EPOCHS", "BATCH_SIZE", "LEARNING_RATE", "WEIGHT_DECAY",
    "WARMUP_FRACTION", "MAX_GRAD_NORM", "SEED", "LIMIT")}
run_info = {"settings": settings, "history": history, "best_epoch": best_epoch}
Path(f"{RUN_DIR}/run_info.json").write_text(json.dumps(run_info, indent=2))

## 10. Quick look at val

A first impression only. The official numbers (with intervals, hard positives, look-alike
negatives, and the regex comparison) come from `evaluate.py`. Test is not looked at here.

In [ ]:
labels = data["val"].label
fpr, tpr, _ = roc_curve(labels, scores["val"])
print(f"val PR AUC: {average_precision_score(labels, scores['val']):.4f} (best epoch {best_epoch})")
for max_fpr in (0.001, 0.01):
    print(f"recall at FPR <= {max_fpr:.1%}: {tpr[fpr <= max_fpr].max():.3f}")